In [ ]:
# Вначале необходимо установить используемые библиотеки.

In [ ]:
!pip list -v

In [ ]:
#!pip install accelerate transformers

In [ ]:
!pip install --upgrade langchain langchain-huggingface langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 950.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found

In [ ]:
!pip install langgraph

In [ ]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=1b1183a5326797d907efcd7867af67a9026a156999ad571d887f9196318df30a
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import ChatHuggingFace
import torch

In [ ]:
# Можно использовать разные модели. Чем больше размер модели, тем в среднем лучше результат.
# Также необходимо помнить что некоторые модели специально дообучаются для использования в качесвте агентов,
# их учат использовать созданные для них инструменты и следовать инструкциям.

# Указываем имя модели и загружаем токенизатор и модель
model_id = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# Создаем pipeline Hugging Face
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.3,
    top_p=0.95,
    repetition_penalty=1.2
)

# На основе pipeline Hugging Face создаем langchain pipeline и langchain chat model
# И то и то можно использовать для построения агентной системы. Основное отличие — структура ввода и вывода.
# pipeline принимает и выдает текст, тогда как chat model работает с запросами в виде диалогов.
llm = HuggingFacePipeline(pipeline=pipe)
chat_model = ChatHuggingFace(llm=llm)

Device set to use cpu


In [ ]:
prompt = "What is the capital of France?"
response = llm.invoke(prompt)
print(response)

In [ ]:
# Ответ chat model содержит специальные теги начала и окончания фразы, метки указывающие кому принадлежит текст
# user - пользовательский контент, assistant - текст модели
# Для размышляющих моделей в диалоге может присутствовать тег think помечающий внутренние размышления модели.
prompt = "What is the capital of France?"
response = chat_model.invoke(prompt)
print(response)

content='<|im_start|>user\nWhat is the capital of France?\no_think<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I need to figure out what the capital of France is. Let me start by recalling any information I have about France and its capitals.\n\nFirst off, I know that France is a country in Europe, known for things like the Eiffel Tower, Paris, and maybe some famous historical events. But wait, when they say "capital," they\'re referring to the city where the government is based, right? So if I remember correctly, Paris is often mentioned as the capital. But let me make sure there\'s not another city that\'s also considered a capital or perhaps a different one.\n\nI think historically, France had several cities that were important, but over time, the seat of government moved. For example, during the French Revolution, the capital was moved from Versailles to Paris because of political reasons. That might be why today Paris is widely recognized as the capital. \n\nBut just to do

In [ ]:
# Построим простейшую RAG (Retrieval-Augmented Generation или генерация, дополненная поиском) систему
# Такая система помогает LLM использовать информацию из внешних источников для формирования ответа, а не только собственные знания.

# В качестве источника дополнительной информации будем использовать простой текстовый документ
from langchain_community.document_loaders import TextLoader

# Загружаем документ (убедитесь что путь указан правильно, также возможны проблемы с кодировкой)
loader = TextLoader("Article.txt", encoding='cp1251')
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")

Loaded 1 document(s).


In [ ]:
# Для разных версий langchain может меняться формат импорта нужных компонент

# from langchain.text_splitter import CharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter

# Большие объемы информации обычно обрабатываются кусками - чанками.
# Это связано с ограничениями контекстного окна моделей.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

print(f"Split into {len(texts)} chunks.")

Split into 23 chunks.


In [ ]:
# Убедимся что информация загружена и нарезана, чанки содержат текст.
print(texts[0].page_content)

Статья доступна по ссылке https://habr.com/ru/articles/861888/?ysclid=mgs27dk7mr171450570

История развития современных нейросетей: хронология, ключевые модели и прорывы


В современных реалиях практически не осталось людей, пропустивших «нейросетевой» шум. Для некоторых, он даже стал фундаментальным инструментом в работе, а кто-то и вовсе ставит его важность наравне с интернетом.

Нейросети плотно начинают входить в нашу жизнь, к счастью, как дружественный инструмент, помогающий повысить точность аналитических выводов. Они используются как простыми людьми для простых задач (помочь распланировать день или отредактировать письмо), так и учёными, в лабораториях, для постановки диагноза, проверки совместимости тех или иных биологических компонентов и т.д.

В сегодняшнем информационном шуме сложно сфокусироваться на истории второстепенных для тебя вещах, поэтому даже самый активный пользователь искусственного интеллекта может не знать откуда «растут корни» - а было бы полезно!


In [ ]:
# Процесс RAG включает несколько этапов:
# Retrieval — поиск и извлечение фрагментов информации из базы знаний, которые наиболее релевантны запросу пользователя.
# Augmentation — дополнение запроса пользователя извлечённой информацией.
# Generation — генерация ответа языковой моделью с учётом дополнительной информации.
# Эти этапы еализованы в классе RetrievalQA
from langchain_classic.chains import RetrievalQA
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

# Сама информация в RAG системе хранится в векторном виде для облегчения поиска релевантных фрагментов.
# Для преобразования в векторный вид используюся Embedd-еры
embeddings = HuggingFaceEmbeddings()

# Преобразуем чанки в вектора, которые будут храниться в памяти.
# В реальной системе в памяти хранятся только наиболее часто используемы чанки,
# основной объем хранится в векторной базе данных.
# LangChain поддерживает различные векторные базы данных, например Chroma,
# Weaviate, Qdrant или база данных Яндекса YDB
vectorstore = InMemoryVectorStore.from_documents(texts, embeddings)

# retrievers сравнивают пользовательский запрос с векторизованными чанками (документами)
# и возвращают наиболее релевантные результаты.
# Для определения релевантности могут использоваться различные метрики, например, косинусная мера или совпадение ключевых слов.
retriever = vectorstore.as_retriever()


# Создадим цепочку формирования ответа на основе RAG
#
# Тип цепи "stuff" все данные от retriever добавляет в контекст запроса к языковой модели.
# Этот подход работает с небольшими фрагментами данных.
# При работе с большим количеством данных используются "refine" или "map_reduce", которые
# позволяют обрабатывать найденные документы каждый по отдельности уточняя и объединяя полученную из них информацию.
qa_chain = RetrievalQA.from_chain_type(
    chat_model,
    retriever=retriever,
    chain_type="stuff"
)

print("Question answering chain created.")


/tmp/ipython-input-1619968894.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipython-input-1619968894.py:8: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/toke

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

NameError: name 'chat_model' is not defined

In [ ]:
# Теперь можно формировать запросы, LLM будет изпользовать загруженные данные для ответа.
# Качество ответов зависит от многих факторов - от используемого Embedd-ера, метрики для поиска релевантных документов,
# поноты информации в самих документах и используемой LLM.
query = 'Какая технологическая компания России наиболее известна своим ИИ?'
result = qa_chain.invoke({"query": query})
print(result['result'])

In [ ]:
# Еще один спостоб построения систем на базе LLM - это агенты.
# Основное отличие агентов - это возможность планирования своих действий и активного взаимодействия
# со специально созданными для них инструментами или другими агентами.

In [ ]:
# В langchain есть встроенные инструменты, предоставляющие агентам возможность
# делать запросы к поисковым движкам или внешним сервисам, выполнять команды в терминале и т.д.
# Создадим инструмент для доступа к Википедии, чтобы агент мог
# верифицировать свой ответ.

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
wikitool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
print(wikitool.invoke({"query": "Kolmogorov"}))

Page: Andrey Kolmogorov
Summary: Andrey Nikolaevich Kolmogorov (Russian: Андре́й Никола́евич Колмого́ров, IPA: [ɐnˈdrʲej nʲɪkɐˈlajɪvʲɪtɕ kəlmɐˈɡorəf] , 25 April 1903 – 20 October 1987) was a Soviet mathematician who played a central role in the creation of modern probability theory. He also contributed to the mathematics of topology, intuitionistic logic, turbulence, classical mechanics, algorithmic information theory and computational complexity.


In [ ]:
# Создадим список инструментов и агента с доступом к ним.
from langgraph.prebuilt import create_react_agent

tools = [wikitool]

agent_executor = create_react_agent(chat_model, tools)

/tmp/ipython-input-1229323681.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(chat_model, tools)


In [ ]:
# Ассистенты обрабатывают сообщения. У сообщений есть системная часть
# в которой описываеся его роль, инструкции
# и возможности использовать инструменты.

# Пользовательский запрос содержится в пользовательской части сообщения.
input_message = {
    "role": "user",
    "content": "Who is Komlogorov?"
}

result = agent_executor.invoke({"messages": [input_message]})
print(result["messages"][1])

content='<|im_start|>system\nТы полезный ассистент. Используй [WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module \'wikipedia\' from \'/usr/local/lib/python3.12/dist-packages/wikipedia/__init__.py\'>, top_k_results=1, lang=\'en\', load_all_available_meta=False, doc_content_chars_max=1000))] для поиска информации и проверки своего ответа. В ответе явно укажи каким инструментом ты пользовался.<|im_end|>\n<|im_start|>user\nWho is Komlogorov?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, the user is asking "Who is Komlogorov?" First, I need to figure out if that\'s a real person or maybe a misspelling. Let me check the name. It could be "Kolmogorov," which I know is a famous mathematician, Andrei Nikolaevich Kolmogorov.\n\nSo, probably the user made a typo. To confirm, I should use the Wikipedia query tool to search for "Komlogorov" and see what comes up. If there\'s no result, it might be a mistake. Alternatively, checking "Kolmogorov" would definitely find the corre

In [ ]:
# Для удобства вывода используется специальный метод
for message in result["messages"]:
  message.pretty_print()

In [ ]:
# Системный промпт можно создать самостоятельно.
system_prompt = (
    f"Ты полезный ассистент. Используй {tools} для поиска информации и проверки своего ответа. В ответе явно укажи каким инструментом ты пользовался."
)

agent_executor = create_react_agent(chat_model, tools, prompt=system_prompt)


/tmp/ipython-input-2013317534.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(chat_model, tools, prompt=system_prompt)


In [ ]:
input_message = {
    "role": "user",
    "content": "Who is Komlogorov?"
}

result = agent_executor.invoke({"messages": [input_message]})

for message in result["messages"]:
  message.pretty_print()

In [ ]:
# Инструменты можно создавать свои.
# Но нужно тщательно формировать описание инструмента, т.к.
# агент сам решает когда его использовать именно на основании описания.
from langchain.tools import tool

@tool("string_invert", description="Check query and return right query")
def string_invert(query: str) -> str:
    return query[::-1]

In [ ]:
tools = [wikitool, string_invert]

# Агенты на основе маленьких LLM часто галюцинируют по поводу использования
# инструментов и могут подменять результаты работы инструмента своими галлюцинациями.
# Для таких LLM нужно четко прописывать системные промпты.

system_prompt = (
    f"Четко следуй инструкциям. Пользовательский запрос может содержать ошибки, его нужно проверять. Для проверки всегда используй {tools[1]}. Используй {tools[0]} для поиска информации. В ответе явно укажи ответ {tools[1]}."
)

agent_executor = create_react_agent(chat_model, tools, prompt=system_prompt)


/tmp/ipython-input-2443594865.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(chat_model, tools, prompt=system_prompt)


In [ ]:
input_message = {
    "role": "user",
    "content": "Who is vorogomloK?"
}

result = agent_executor.invoke({"messages": [input_message]})

for message in result["messages"]:
  message.pretty_print()

================================ Human Message =================================

Who is vorahkaS?
================================== Ai Message ==================================

<|im_start|>system
Четко следуй инструкциям. Пользовательский запрос может содержать ошибки, его нужно проверять. Для проверки всегда используй name='string_invert' description='Check query and return right query' args_schema=<class 'langchain_core.utils.pydantic.string_invert'> func=<function string_invert at 0x78be606d6d40>. Используй api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/usr/local/lib/python3.12/dist-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=1000) для поиска информации. В ответе явно укажи ответ name='string_invert' description='Check query and return right query' args_schema=<class 'langchain_core.utils.pydantic.string_invert'> func=<function string_invert at 0x78be606d6d40>.<|im_end|>
<|im_start|>us

In [ ]:
# Удачи в построении интеллектуальных систем!